# AutoARIMAParams -- Engine Test Notebook

Tests the updated `AutoARIMAParams` / `ARIMAXConfig` engine before wiring into the UI.

**Sections**
1. Setup and import check
2. Inspect `AutoARIMAParams` defaults and the 7 parameter groups
3. Synthetic dataset generation
4. Baseline run (all defaults -- identical to old engine)
5. Customised runs: BIC, random search, grid search, solver, intercept, ADF
6. Direct `run_one_combo` test with forecast plot
7. Side-by-side metrics comparison
8. Full parameter reference table

---

## 1. Setup & import check

In [1]:
import sys, pathlib

# Add project root to sys.path so imports work when opened directly.
ROOT = pathlib.Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

Project root: C:\Users\Limu\Documents\Resume\TMT Research 半导体集成电路\stock_engine


In [2]:
import warnings
import numpy as np
import pandas as pd

from stock_engine.analytics.arimax import (
    ARIMAXConfig, ARIMAXPipeline, AutoARIMAParams,
)
from stock_engine.analytics.arimax.pipeline import _build_auto_arima_kwargs
print('All engine imports OK')

All engine imports OK


In [3]:
import pmdarima as pm
print(f'pmdarima version: {pm.__version__}')

pmdarima version: 2.1.1


---
## 2. Inspect AutoARIMAParams defaults

In [4]:
p = AutoARIMAParams()

GROUPS = {
    'Order bounds':    ['start_p','d','start_q','max_p','max_d','max_q'],
    'Seasonal':        ['start_P','D','start_Q','max_P','max_D','max_Q','max_order','m','seasonal'],
    'Stationarity':    ['stationary','information_criterion','alpha','test','seasonal_test'],
    'Search strategy': ['stepwise','n_jobs','random','random_state','n_fits'],
    'Fitting':         ['method','maxiter','with_intercept','trend'],
    'Validation':      ['out_of_sample_size','scoring','scoring_args'],
    'Behaviour':       ['suppress_warnings','error_action'],
}

rows = []
for group, fields in GROUPS.items():
    for f in fields:
        rows.append({'Group': group, 'Parameter': f, 'Default': getattr(p, f)})

df_params = pd.DataFrame(rows).set_index(['Group','Parameter'])
print(f'Total parameters: {len(df_params)}')
df_params

Total parameters: 34


Default
Group           Parameter                    
Order bounds    start_p                     2
                d                        None
                start_q                     2
                max_p                       5
                max_d                       2
                max_q                       5
Seasonal        start_P                     1
                D                        None
                start_Q                     1
                max_P                       2
                max_D                       1
                max_Q                       2
                max_order                   5
                m                           1
                seasonal                 True
Stationarity    stationary              False
                information_criterion     aic
                alpha                    0.05
                test                     kpss
                seasonal_test            ocsb
Search strategy stepwise                 True
                n_jobs                      1
                random                  False
                random_state             None
                n_fits                     10
Fitting         method                  lbfgs
                maxiter                    50
                with_intercept           auto
                trend                    None
Validation      out_of_sample_size          0
                scoring                   mse
                scoring_args             None
Behaviour       suppress_warnings        True
                error_action           ignore

In [5]:
kw = _build_auto_arima_kwargs(p, trace=False)
print(f'kwargs count : {len(kw)}  (34 expected -- 33 params + trace)')
print(f'sorted keys  : {sorted(kw.keys())}')
print()
kw_ns = _build_auto_arima_kwargs(p, trace=False, force_nonseasonal=True)
print(f'seasonal (normal)           : {kw["seasonal"]}')
print(f'seasonal (force_nonseasonal): {kw_ns["seasonal"]}')

kwargs count : 34  (34 expected -- 33 params + trace)
sorted keys  : ['D', 'alpha', 'd', 'error_action', 'information_criterion', 'm', 'max_D', 'max_P', 'max_Q', 'max_d', 'max_order', 'max_p', 'max_q', 'maxiter', 'method', 'n_fits', 'n_jobs', 'out_of_sample_size', 'random', 'random_state', 'scoring', 'seasonal', 'seasonal_test', 'start_P', 'start_Q', 'start_p', 'start_q', 'stationary', 'stepwise', 'suppress_warnings', 'test', 'trace', 'trend', 'with_intercept']

seasonal (normal)           : True
seasonal (force_nonseasonal): False


---
## 3. Synthetic dataset

80 monthly observations:
- **Target**: simulated monthly portfolio excess return (AR(1) with noise)
- **CPI**: level series (log_diff gives monthly growth rate)
- **Fed rate**: level series (diff gives monthly bp change)

In [6]:
rng = np.random.default_rng(42)
N   = 80
idx = pd.date_range('2018-01-31', periods=N, freq='ME')

# Target: AR(1) excess return
y = np.zeros(N)
y[0] = 0.005
for t in range(1, N):
    y[t] = 0.3 * y[t-1] + rng.normal(0.004, 0.025)

# CPI level
cpi = 100 * np.cumprod(1 + rng.normal(0.003, 0.001, N))

# Fed funds rate (AR(1) bounded)
fed = np.zeros(N)
fed[0] = 2.5
for t in range(1, N):
    fed[t] = float(np.clip(0.95 * fed[t-1] + rng.normal(0, 0.15), 0.1, 8.0))

raw = pd.DataFrame({'portfolio_return': y, 'cpi': cpi, 'fed_rate': fed}, index=idx)
print(raw.shape)
raw.head(6)

(80, 3)


,portfolio_return,cpi,fed_rate
2018-01-31,0.005000,100.269065,2.500000
2018-02-28,0.013118,100.615673,2.197258
2018-03-31,-0.018064,100.850920,1.942628
2018-04-30,0.017342,101.116858,1.736713
2018-05-31,0.032717,101.381609,1.969148
2018-06-30,-0.034961,101.564517,1.747482


---
## 4. Baseline run -- all defaults

This should produce **identical results** to the old hard-coded engine.
`AutoARIMAParams()` defaults match pmdarima exactly.

In [7]:
cfg_default = ARIMAXConfig(
    target      = 'portfolio_return',
    mevs        = ['cpi', 'fed_rate'],
    mevs_guide  = {'cpi': 'log_diff', 'fed_rate': 'diff'},
    freq        = 'ME',
    max_lag     = 2,
    max_number_of_exogenous_variables = 2,
    train_threshold = 0.8,
    transform   = 'original',
    trace       = False,
    # arima_params not set --> AutoARIMAParams() defaults applied
)
print('arima_params:', cfg_default.arima_params)

arima_params: AutoARIMAParams(start_p=2, d=None, start_q=2, max_p=5, max_d=2, max_q=5, start_P=1, D=None, start_Q=1, max_P=2, max_D=1, max_Q=2, max_order=5, m=1, seasonal=True, stationary=False, information_criterion='aic', alpha=0.05, test='kpss', seasonal_test='ocsb', stepwise=True, n_jobs=1, random=False, random_state=None, n_fits=10, method='lbfgs', maxiter=50, with_intercept='auto', trend=None, out_of_sample_size=0, scoring='mse', scoring_args=None, suppress_warnings=True, error_action='ignore')


In [8]:
%%time
warnings.filterwarnings('ignore')

pipe_default = ARIMAXPipeline(config=cfg_default)
pipe_default.set_raw(raw)
pipe_default.preprocess(ds_column='date')
pipe_default.build_combinations(ds_column='date')
pipe_default.split_data()
pipe_default.run_models(use_tqdm=True)   # <-- progress bar per rolling-forecast step
pipe_default.apply_hard_rules()
top_default  = pipe_default.rank_top_models()

print(f'Models fitted    : {len(pipe_default.results_df)}')
n_hard = len(pipe_default.results_after_hard) if pipe_default.results_after_hard is not None else 'N/A'
print(f'After hard rules : {n_hard}')
top_default[['X_combo','order','mse','mae','rmse','mape']]

Models fitted    : 9
After hard rules : 3


KeyError: "['mse', 'mae', 'rmse', 'mape'] not in index"

---
## 5. Customised runs

Each sub-section changes one parameter group.

### 5-A  BIC criterion + wider order search (max_p=8, max_q=8)

In [9]:
%%time
cfg_bic = ARIMAXConfig(
    target='portfolio_return', mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    freq='ME', max_lag=2, train_threshold=0.8,
    arima_params=AutoARIMAParams(
        information_criterion = 'bic',
        max_p = 8,
        max_q = 8,
    ),
)
top_bic = ARIMAXPipeline(config=cfg_bic).run(raw, ds_column='date')
top_bic[['X_combo','order','mse','mae']]

KeyError: "['mse', 'mae'] not in index"

### 5-B  Random search (random=True, n_fits=15, random_state=42)

Samples 15 random ARIMA configs instead of stepwise traversal.

In [ ]:
%%time
cfg_random = ARIMAXConfig(
    target='portfolio_return', mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    freq='ME', max_lag=2, train_threshold=0.8,
    arima_params=AutoARIMAParams(
        stepwise     = False,
        random       = True,
        random_state = 42,
        n_fits       = 15,
    ),
)
top_rnd = ARIMAXPipeline(config=cfg_random).run(raw, ds_column='date')
top_rnd[['X_combo','order','mse','mae']]

### 5-C  Exhaustive grid search (stepwise=False)

Evaluates every ARIMA(p,d,q) up to max_order. Slower but finds the global optimum.

In [ ]:
%%time
cfg_grid = ARIMAXConfig(
    target='portfolio_return', mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    freq='ME', max_lag=2, train_threshold=0.8,
    arima_params=AutoARIMAParams(
        stepwise  = False,
        max_p     = 3,
        max_q     = 3,
        max_order = 4,
    ),
)
top_grid = ARIMAXPipeline(config=cfg_grid).run(raw, ds_column='date')
top_grid[['X_combo','order','mse','mae']]

### 5-D  Newton solver + more iterations

In [ ]:
%%time
cfg_newton = ARIMAXConfig(
    target='portfolio_return', mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    freq='ME', max_lag=2, train_threshold=0.8,
    arima_params=AutoARIMAParams(method='newton', maxiter=100),
)
top_newton = ARIMAXPipeline(config=cfg_newton).run(raw, ds_column='date')
top_newton[['X_combo','order','mse','mae']]

### 5-E  No intercept (with_intercept=False)

In [ ]:
%%time
cfg_noint = ARIMAXConfig(
    target='portfolio_return', mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    freq='ME', max_lag=2, train_threshold=0.8,
    arima_params=AutoARIMAParams(with_intercept=False),
)
top_noint = ARIMAXPipeline(config=cfg_noint).run(raw, ds_column='date')
top_noint[['X_combo','order','mse','mae']]

### 5-F  ADF unit-root test + tighter alpha (0.01)

In [ ]:
%%time
cfg_adf = ARIMAXConfig(
    target='portfolio_return', mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    freq='ME', max_lag=2, train_threshold=0.8,
    arima_params=AutoARIMAParams(test='adf', alpha=0.01),
)
top_adf = ARIMAXPipeline(config=cfg_adf).run(raw, ds_column='date')
top_adf[['X_combo','order','mse','mae']]

---
## 6. Direct run_one_combo test

In [ ]:
from stock_engine.analytics.arimax.model import run_one_combo
from stock_engine.analytics.arimax.preprocess import data_preprocess, train_test_split

df_proc = data_preprocess(
    raw=raw, target='portfolio_return',
    mevs=['cpi','fed_rate'],
    mevs_guide={'cpi':'log_diff','fed_rate':'diff'},
    ds_column='date', freq='ME', max_lag=2,
)
split = train_test_split(df_proc, target='portfolio_return', train_threshold=0.8)

print(f'Train: {len(split.y_train)} obs   Test: {len(split.y_test)} obs')
exog_cols = [c for c in df_proc.columns if c not in ['portfolio_return','date']]
print(f'Exog cols: {exog_cols}')

In [ ]:
%%time
row, preds = run_one_combo(
    combo_cols   = ['cpi_lag1'],
    y_train      = split.y_train,
    y_test       = split.y_test,
    train_data   = split.train_data,
    test_data    = split.test_data,
    transform    = 'original',
    trace        = True,
    use_tqdm     = True,
    arima_params = AutoARIMAParams(
        max_p = 4, max_q = 4,
        information_criterion = 'bic',
    ),
)

print(f"Order : {row['order']}")
print(f"MSE   : {row['mse']:.6f}")
print(f"MAE   : {row['mae']:.6f}")
print(f"RMSE  : {row['rmse']:.6f}")
print(f"MAPE  : {row['mape']:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as _np

test_dates = raw.index[-len(split.y_test):]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_dates, split.y_test,       label='Actual',   lw=1.5, color='steelblue')
ax.plot(test_dates, preds['forecasts'], label='Forecast', lw=1.5, linestyle='--', color='orange')
confs = _np.array(preds['confs'])
ax.fill_between(test_dates, confs[:,0], confs[:,1], alpha=0.15, color='orange', label='95% CI')
ax.axhline(0, color='grey', lw=0.6, linestyle=':')
ax.set_title(f"cpi_lag1 | order={row['order']} | BIC")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Side-by-side metrics comparison

In [ ]:
def _top1(top_df, label):
    if top_df is None or top_df.empty:
        return {'run': label, 'combo': 'N/A', 'order': 'N/A',
                'mse': None, 'mae': None, 'rmse': None, 'mape': None}
    r = top_df.iloc[0]
    return {
        'run'  : label,
        'combo': r['X_combo'],
        'order': str(r['order']),
        'mse'  : round(r['mse'],  6),
        'mae'  : round(r['mae'],  6),
        'rmse' : round(r['rmse'], 6),
        'mape' : round(r['mape'], 2),
    }

summary = pd.DataFrame([
    _top1(top_default, 'Default (AIC, stepwise)'),
    _top1(top_bic,     'BIC  max_p=8 max_q=8'),
    _top1(top_rnd,     'Random search n_fits=15'),
    _top1(top_grid,    'Grid search (no stepwise)'),
    _top1(top_newton,  'Newton solver maxiter=100'),
    _top1(top_noint,   'No intercept'),
    _top1(top_adf,     'ADF test alpha=0.01'),
]).set_index('run')
summary

---
## 8. Full parameter reference

In [ ]:
REF = [
    # (group, param, default, type, options, note)
    ('Order bounds',  'start_p', 2,    'int',      '0 to max_p',           'Starting AR order'),
    ('Order bounds',  'd',       None, 'int|None', '0/1/2 or None',        'None = auto via test'),
    ('Order bounds',  'start_q', 2,    'int',      '0 to max_q',           'Starting MA order'),
    ('Order bounds',  'max_p',   5,    'int',      '1 to 20',              'Max AR order'),
    ('Order bounds',  'max_d',   2,    'int',      '0 to 3',               'Max differencing'),
    ('Order bounds',  'max_q',   5,    'int',      '1 to 20',              'Max MA order'),
    ('Seasonal',      'seasonal',True, 'bool',     'True/False',           'Forced False for log/boxcox'),
    ('Seasonal',      'm',       1,    'int',      '1/4/12/52/365',        '1=none 4=Q 12=M 52=W'),
    ('Seasonal',      'start_P', 1,    'int',      '0 to max_P',           'Starting seasonal AR'),
    ('Seasonal',      'D',       None, 'int|None', '0/1 or None',          'None = auto'),
    ('Seasonal',      'start_Q', 1,    'int',      '0 to max_Q',           'Starting seasonal MA'),
    ('Seasonal',      'max_P',   2,    'int',      '0 to 5',               'Max seasonal AR'),
    ('Seasonal',      'max_D',   1,    'int',      '0 to 2',               'Max seasonal differencing'),
    ('Seasonal',      'max_Q',   2,    'int',      '0 to 5',               'Max seasonal MA'),
    ('Seasonal',      'max_order',5,   'int|None', '1 to 20 or None',      'Max p+q+P+Q (grid search)'),
    ('Stationarity',  'stationary',False,'bool',   'True/False',           'Force d=0'),
    ('Stationarity',  'information_criterion','aic','str','aic/bic/hqic/oob','Model selection criterion'),
    ('Stationarity',  'alpha',  0.05,  'float',    '0.01 to 0.10',         'Unit-root test level'),
    ('Stationarity',  'test',   'kpss','str',      'kpss/adf/pp',          'Stationarity test for d'),
    ('Stationarity',  'seasonal_test','ocsb','str','ocsb/ch',              'Seasonal test for D'),
    ('Search',        'stepwise',True, 'bool',     'True/False',           'Stepwise vs full grid'),
    ('Search',        'n_jobs',  1,    'int',      '1 to cpu_count',       'Parallel jobs (grid only)'),
    ('Search',        'random',  False,'bool',     'True/False',           'Random search over space'),
    ('Search',        'random_state',None,'int|None','any int or None',    'PRNG seed'),
    ('Search',        'n_fits',  10,   'int',      '5 to 200',             'Configs tried when random'),
    ('Fitting',       'method', 'lbfgs','str',     'lbfgs/bfgs/newton/nm/cg/ncg/powell','scipy solver'),
    ('Fitting',       'maxiter', 50,   'int',      '20 to 500',            'Optimiser iterations'),
    ('Fitting',       'with_intercept','auto','any','True/False/auto',     'auto adapts during search'),
    ('Fitting',       'trend',   None, 'str|None', 'n/c/t/ct or None',     'Trend component'),
    ('Validation',    'out_of_sample_size',0,'int','0 to 20% of series',   'Hold-out obs for scoring'),
    ('Validation',    'scoring','mse', 'str',      'mse/mae',              'OOS scoring metric'),
    ('Validation',    'scoring_args',None,'dict|None','None or dict',      'Extra scoring kwargs'),
    ('Behaviour',     'suppress_warnings',True,'bool','True/False',        'Suppress statsmodels warnings'),
    ('Behaviour',     'error_action','ignore','str','warn/raise/ignore/trace','On-fit-failure action'),
]

df_ref = pd.DataFrame(REF, columns=['Group','Parameter','Default','Type','Options','Note'])
df_ref = df_ref.set_index(['Group','Parameter'])
print(f'{len(df_ref)} parameters total')
df_ref

---
## Notes for next steps

### Suggested UI grouping (7 sections)

| Section | Key params | Suggested control |
|---|---|---|
| Order bounds | start_p max_p start_q max_q max_d d | number inputs + toggle for d=auto |
| Seasonal | seasonal m start/max P Q D | toggle gate; sub-controls appear when seasonal=True |
| Stationarity | information_criterion test alpha | selectbox + slider |
| Search strategy | stepwise n_jobs random n_fits random_state | radio (stepwise/grid/random) + conditional |
| Fitting | method maxiter with_intercept trend | selectboxes + number input |
| Validation | out_of_sample_size scoring | number input + selectbox |
| Behaviour | suppress_warnings error_action | checkboxes + selectbox |

### Params safe to hide in an 'Advanced' expander
- `start_p`, `start_q`, `start_P`, `start_Q` (rarely need changing)
- `trend`, `with_intercept` (edge cases)
- `scoring_args` (dict, hard to input via UI)
- `out_of_sample_size` (pipeline already does train/test split)
- `n_jobs`, `random_state` (infra-level)

### Params recommended for the primary panel
- `information_criterion` (AIC vs BIC is a key analyst choice)
- `max_p`, `max_q` (model complexity budget)
- `seasonal` + `m` (enable with period picker: 1 / 4 / 12)
- `stepwise` / `random` toggle
- `test` (stationarity test: kpss / adf)
- `d` (force a fixed value or leave as None for auto-detect)